# Local Serving Smoke Test

This notebook tests the local `FastAPI + ONNX` serving path against the already-running local stack.

Start the stack first from a host terminal:

```bash
cd serving
docker compose -f docker-compose-local-test.yaml up --build -d
```

Expected outcomes:

1. Confirm the repo is mounted into Jupyter.
2. Verify `GET /health` on the serving API.
3. Send a sample `POST /predict` request.
4. Confirm `Mealie` is reachable on `localhost:9000`.


In [ ]:
import json
import time
from pathlib import Path
from urllib import request as urllib_request
from urllib.error import URLError, HTTPError

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
NOTEBOOK_PATH = REPO_ROOT / 'notebooks' / 'local_serving_smoke_test.ipynb'
SERVING_URL = 'http://substitution-serving:8000'
MEALIE_URL = 'http://mealie:9000'

assert NOTEBOOK_PATH.exists(), f'Missing notebook file: {NOTEBOOK_PATH}'
print('Repo root:', REPO_ROOT)
print('Notebook file:', NOTEBOOK_PATH)
print('Serving URL:', SERVING_URL)
print('Mealie URL:', MEALIE_URL)

def http_json(method, url, payload=None):
    body = None
    headers = {}
    if payload is not None:
        body = json.dumps(payload).encode('utf-8')
        headers['Content-Type'] = 'application/json'
    req = urllib_request.Request(url, data=body, headers=headers, method=method)
    with urllib_request.urlopen(req, timeout=10) as resp:
        data = resp.read().decode('utf-8')
        return resp.status, json.loads(data)

def http_status(url):
    req = urllib_request.Request(url, method='GET')
    with urllib_request.urlopen(req, timeout=10) as resp:
        return resp.status, resp.headers.get_content_type()


In [ ]:
deadline = time.time() + 120
last_error = None
while time.time() < deadline:
    try:
        status, data = http_json('GET', f'{SERVING_URL}/health')
        print('Serving health status:', status)
        print(json.dumps(data, indent=2))
        break
    except (URLError, HTTPError, json.JSONDecodeError, TimeoutError, ConnectionError) as exc:
        last_error = exc
        time.sleep(2)
else:
    raise RuntimeError(f'Serving API never became healthy: {last_error}')


In [ ]:
payload = {
    'recipe_context': [12, 45, 3, 88],
    'missing_ingredient': 77,
}
status, prediction = http_json('POST', f'{SERVING_URL}/predict', payload)
print('Predict status:', status)
print(json.dumps(prediction, indent=2))
assert 'substitutions' in prediction and len(prediction['substitutions']) > 0


In [ ]:
deadline = time.time() + 120
last_error = None
while time.time() < deadline:
    try:
        status, content_type = http_status(MEALIE_URL)
        print('Mealie status:', status)
        print('Mealie content type:', content_type)
        break
    except (URLError, HTTPError, TimeoutError, ConnectionError) as exc:
        last_error = exc
        time.sleep(2)
else:
    raise RuntimeError(f'Mealie never became reachable: {last_error}')


## Browser URLs

1. Jupyter Lab: `http://localhost:8899`
2. Serving health: `http://localhost:8000/health`
3. Mealie: `http://localhost:9000`

When you are done, stop the stack from a host terminal:

```bash
cd serving
docker compose -f docker-compose-local-test.yaml down
```
